In [1]:
import numpy as np
from typing import List
import random
from microlensing import microlensing
import pandas as pd
import matplotlib.pyplot as plt
import utils as ms_utils
from utils import Param


In [2]:
def plot_2d_contours(chis, params_comb):
    chis=np.array(chis)
    u_min, T0 = zip(*params_comb)
    levels = np.array([0,2.30,4.61,6.17])+min(chis)
    fig, ax2 = plt.subplots()
    cs = ax2.tricontour(u_min,T0,chis, levels=levels, linewidths=0.5,colors=('red',  'green', 'orange'))
    #cntr2 = ax2.tricontourf(u_min,T0,chis, levels=levels, cmap='Blues')
    ax2.clabel(cs, inline=True, fontsize=10)

    #fig.colorbar(cntr2, ax=ax2)
    #ax2.plot(u_min,T0, 'ko', ms=1)
    ax2.set_title('u_min vs. T0 effect on Goodness of Fit (chi)')

    plt.show()

def chis_for_contours(chis, params_comb, ypoints):
    fit_params = params_comb[np.argmin(chis)] #הפרמטרים המינימלים
    df = pd.DataFrame(params_comb)
    df['chis'] = np.array(chis)
    min_chi = min(df['chis'])
    df = df[df['chis']<min_chi+13].drop(columns=['chis'])
    bounds= df.max()
    params: List[Param] = [] 
    for i in range(len(fit_params)):
        res = abs(fit_params[i]-bounds[i])
        param = Param(fit_params[i]-res,fit_params[i]+res,27)
        params.append(param)
    n_params, chis , params_combinations=ms_utils.min_chi_on_params(ypoints, params)
    return chis , params_combinations


def params_errors(chis, params_combinations):
    fit_params = dict(enumerate(params_combinations[np.argmin(chis)].flatten()))
    min_chi = min(chis)
    df = pd.DataFrame(params_combinations)
    df['chis'] = np.array(chis)
    #print(df[df['chis']<min_chi+1])
    df = df[df['chis']>min_chi+1]
    error_df = pd.DataFrame(columns=['value', 'max_bound', 'min_bound', 'up_error' , 'down_error'])
    for p_index in fit_params:
        const_params = fit_params.copy()
        const_params.pop(p_index)
        temp_df=df
        for key in const_params:    # make all other params const on their min chi value
            temp_df=temp_df[temp_df[key]==const_params[key]] 
        big_df = temp_df[temp_df[p_index]>fit_params[p_index]]
        small_df = temp_df[temp_df[p_index]<fit_params[p_index]]
        min_bound = small_df[small_df['chis']==small_df['chis'].min()].reset_index()[p_index][0]
        max_bound = big_df[big_df['chis']==big_df['chis'].min()].reset_index()[p_index][0]
        down_error = fit_params[p_index]- min_bound
        up_error = max_bound - fit_params[p_index]
        error_df.loc[p_index] = [fit_params[p_index],max_bound,min_bound,up_error,down_error]
        #       print (p_index , p )
     #       print(const_params)
    return error_df

def plot_non_linear_fit(ypoints, params):
    f_x = ms_utils.do_func(params , ypoints.x)
    plt.errorbar(x = ypoints.x, y = ypoints.y, yerr = ypoints.err, fmt = 'o', markersize = 0.5)
    plt.plot(ypoints.x,f_x)
    plt.xlabel('T0 (days) -2458000')
    plt.ylabel('I/I_0')
    plt.title('plot of 2D non linear fit')
    plt.grid()
    plt.show()

    plt.errorbar(x = ypoints.x, y = ypoints.y-f_x, yerr = ypoints.err, fmt='o', markersize=2)
    plt.xlabel('T0 (days) -2458000')
    plt.ylabel('yi-f(xi)')
    plt.axhline(y = 0, linestyle = '--')
    plt.title('residuals plot for 2D non linear fit')
    plt.grid()
    plt.show()

def print_nsigma(ogle_name,fit_val):
        print("")
        print(f"nsigma with ogle {ogle_name} parameter")
        print(ms_event.ogle[ogle_name])
        print(fit_val)
        print(f"nsigma: {ms_utils.nsigma(ms_event.ogle[ogle_name], fit_val)}")
        print("\n")


In [3]:
ms_event = microlensing("https://www.astrouw.edu.pl/ogle/ogle4/ews/2019/blg-0035")
ms_event.data['norm_time'] =ms_event.data['JHD']-2458000

In [4]:
params = [Param("u_min",0.677,0.681), Param("t0",590.1,590.6)]
fixed_params = [Param("tau",60.799,60.799,1), Param("f_bl",1.0,1.0,1)]
ypoints = ms_event.data[['norm_time','I','I_error']].set_axis(['x','y','err'],axis=1)

In [10]:
non_linear_2d_fit = ms_utils.MeshgridChiMinNonLinearFit(ypoints.x, ypoints.y, ypoints.err, ms_utils.do_func)
all_chis, all_params_comb  = non_linear_2d_fit.fit(params, fixed_params)

In [ ]:
min_chi_params = all_params_comb[np.argmin(all_chis)]
plot_non_linear_fit(ypoints, min_chi_params)

In [7]:
chis , params_combinations = chis_for_contours(all_chis, all_params_comb, ypoints)

In [ ]:
plot_2d_contours(chis, params_combinations)

In [ ]:
error_df = params_errors(chis , params_combinations)
print(error_df)

In [ ]:
a_s_list = np.loadtxt('a_s_list1.txt', delimiter=",")
chi_list = np.loadtxt('chi_list1.txt', delimiter=",")

In [ ]:
u_min_list , T0_list = zip(*a_s_list)

In [ ]:
T0_hist = ms_utils.norm_hist('T0',T0_list)

In [ ]:
u_min_hist = ms_utils.norm_hist('u_min',u_min_list)

In [ ]:
u_min_fit = ms_utils.value_with_error('u_min fit',error_df.loc[0].value , error_df.loc[0].up_error)
T0_fit = ms_utils.value_with_error('T0 fit',error_df.loc[1].value , error_df.loc[1].up_error)

In [ ]:
ms_utils.bootstrap_compare(T0_fit, T0_hist)
print('\n')
ms_utils.bootstrap_compare(u_min_fit, u_min_hist)

In [ ]:
print_nsigma('umin',u_min_fit)
T0_fit.value = T0_fit.value+2458000
print_nsigma('t0',T0_fit)